In [1]:
from utils.orthanc_utils import *
from utils.db_utils import *
from utils.plot_utils import *
# from utils.pipeline import *

In [3]:
db = TinyDB(DB_PATH)
studies = fetch_db_studies()

study = studies[0]

In [4]:
df = pd.DataFrame([series.__dict__ for series in study.series_dict.values()])
sax_dl_df = df[(df['dl_orthanc_id'].notna()) & (df['roundel_orthanc_id'].isna())]

image_instances = fetch_orthanc_instances_for_series_list(sax_dl_df["orthanc_series_id"].dropna().unique()) 
mask_instances = fetch_orthanc_instances_for_series_list(sax_dl_df["dl_orthanc_id"].dropna().unique())

'0161e469-20164632-d5fdc1b9-aae50214-6e0bef97'

In [74]:
sax_df, image_4d = get_4d_array(image_instances)
_, mask_4d = get_4d_array(mask_instances)

In [ ]:
image_flat = [
    image_4d[:, :, sl, t]
    for sl in range(image_4d.shape[2])
    for t in range(image_4d.shape[3])
]

In [76]:
new_sax_df = sax_df.copy()
new_sax_df['PixelArray'] = image_flat

In [81]:
for series_id, series_df in sax_df.groupby('OrthancSeriesID'):
    old_dcms = [fetch_orthanc_dicom(id) for id in series_df.OrthancInstanceID]
    new_masks = [mask for mask in series_df.PixelArray]
    send_series_to_orthanc(new_masks, old_dcms, new_description='Roundel')